# 01b: Drop: Place a held box back on the table (warm-started from walk)

| | |
|---|---|
| **Project** | Humanoid Loco-Manipulation with the Unitree G1 |
| **Developer** | Narayane Ribeiro Medeiros |
| **Repository** | [https://github.com/narayanerm/Humanoid-Loco-Manipulation](https://github.com/narayanerm/Humanoid-Loco-Manipulation) |

Trains a **single G1 policy** that both picks a box off a table AND places a held box back on the table. Each reset samples either scenario (curriculum grows the place fraction over training). Same reward machinery, bidirectional dual-Gaussian on box-to-target distance, `target_progress` (proximity, not height), and lowered `hold_box` weight so the physics does the release naturally.


### Run-Mode Flags
Boolean switches: reuse a pick checkpoint, resume, warm-start from the walk policy, run a smoke test, or open the viewer.

In [ ]:
REUSE_TRAINING        = False  # skip training if PPO checkpoint exists
RESUME                = False  # continue from the most recent PPO checkpoint
WARM_START_FROM_REACH = False   # load reach checkpoint as initialization (hierarchical-style HRL)
RESET_STD_ON_RESUME   = 1.0    # on RESUME, force actor Gaussian std to this scalar (None = keep loaded std)
DEBUG_LIGHT           = True   # True = 64 envs / False = 8192 envs, few iters
RUN_VIEWER            = True  # open viser viewer at localhost:8080

### Environment Setup
Install pinned dependencies from `requirements.txt` via `uv`, configure MuJoCo in headless mode (EGL), and detect the training GPU.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 required but got {sys.version.split()[0]}')

os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ['UV_HTTP_TIMEOUT'] = '600'

REQ = Path('../requirements.txt').resolve()
if not REQ.exists():
    raise FileNotFoundError(f'requirements.txt not found at {REQ}')

if shutil.which('uv') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

subprocess.check_call([
    sys.executable, '-m', 'uv', 'pip', 'install',
    '--index-strategy', 'unsafe-best-match',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    '-r', str(REQ),
])

import torch, mjlab
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f'GPU count : {n}')
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {p.name}  |  {p.total_memory / 1e9:.1f} GB  |  sm_{p.major}{p.minor}')
    DEVICE = 'cuda:0'
    torch.cuda.set_device(0)
else:
    DEVICE = 'cpu'
    print('WARNING: no GPU detected')

try:
    subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.free',
                    '--format=csv,noheader'], check=False)
except FileNotFoundError:
    pass


### Robot Initialization
Loads `TASK_ID`, G1 joint list, and home pose plus table/box geometry from `src/policy_drop.py`.

In [ ]:
import sys, os
from pathlib import Path
for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (p / 'LICENSE').exists():
        os.chdir(p); break
sys.path.insert(0, str(Path.cwd()))


from src.policy_drop import (
    TASK_ID,
    G1_JOINTS, G1_HOME_ROOT_POS, G1_HOME_ROOT_ROT, G1_HOME_JOINT_POS,
    TABLE_X, TABLE_TOP_Z, BOX_HALF, BOX_START_X, BOX_REST_Z,
    CHEST_Z, PLACE_TARGET_Z_CENTER,
)

print(f'Task ID          : {TASK_ID}')
print(f'Joints           : {len(G1_JOINTS)}  (derived from mjlab G1 spec)')
print(f'Home root z      : {G1_HOME_ROOT_POS[2]:.3f} m')
print()
print(f'Table top z      : {TABLE_TOP_Z:.3f} m  (front edge at x={TABLE_X - 0.34:.2f})')
print(f'Box rest         : ({BOX_START_X:.2f}, 0.00, {BOX_REST_Z:.3f})  (half-size {BOX_HALF:.3f} m, box centre at rest on table)')
print()
print(f'Chest height     : {CHEST_Z:.3f} m  (carry / transit target)')
print(f'Place target z   : {PLACE_TARGET_Z_CENTER:.3f} m  (box back on table)')


### Task Config & Registration
The full MDP (observations, actions, rewards, events, terminations, scene entities) plus the PPO runner cfg live in `src/policy_drop.py`. Edit that file to change rewards, geometry, curriculum, or PPO hyperparameters.

In [ ]:
from src import policy_drop

policy_drop.register()

env_cfg = policy_drop.g1_drop_env_cfg(play=False)
print(f'Task registered  : {policy_drop.TASK_ID}')
print(f'Entities         : {list(env_cfg.scene.entities.keys())}')
print(f"Sensors          : {len(env_cfg.scene.sensors)}  ({', '.join(s.name for s in env_cfg.scene.sensors)})")
print(f'Events           : {list(env_cfg.events.keys())}')
print(f'Rewards          : {len(env_cfg.rewards)} terms   -> {list(env_cfg.rewards.keys())}')
print(f'Terminations     : {list(env_cfg.terminations.keys())}')
print(f'Curriculum       : {list(env_cfg.curriculum.keys())}')
print(f'Sim iter/ls      : {env_cfg.sim.mujoco.iterations}/{env_cfg.sim.mujoco.ls_iterations}')
print(f'Episode length   : {env_cfg.episode_length_s:.2f} s   ({int(env_cfg.episode_length_s / 0.025)} steps @ 40 Hz)')


### Simulation Parameters
Global run settings shared by training and play: task, parallel envs, PPO hyperparameters, and W&B logging. `DEBUG_LIGHT` shrinks to 64 envs / 3 iters for a smoke test.

In [ ]:
from pathlib import Path
import sys; sys.path.insert(0, str(Path('.').resolve()))
from src.utils import find_latest_checkpoint

OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
MEDIA = OUT / 'media'; MEDIA.mkdir(exist_ok=True)

EXPERIMENT_HINT       = 'g1_drop'
# Warm-start from carry: has grip prior needed for drop.
REACH_EXPERIMENT_HINT = 'g1_carry'
LOG_ROOT              = str(OUT / 'logs')

NUM_ENVS         = 8192
MAX_ITERATIONS   = 4000

SIM_TIMESTEP     = 0.005
DECIMATION       = 5
EPISODE_LENGTH_S = 3.25

LEARNING_RATE    = 3e-4
CLIP_PARAM       = 0.2
GAMMA            = 0.97
LAMBDA_GAE       = 0.95
ENTROPY_COEF     = 0.01

if DEBUG_LIGHT:
    NUM_ENVS, MAX_ITERATIONS = 64, 3

print(f'Task            : {TASK_ID}')
print(f'Parallel envs   : {NUM_ENVS}')
print(f'PPO iterations  : {MAX_ITERATIONS}')
print(f'Policy freq     : {1 / (SIM_TIMESTEP * DECIMATION):.0f} Hz')
print(f'Episode length  : {EPISODE_LENGTH_S:.2f} s')
print(f'Outputs dir     : {OUT.resolve()}')
print(f'Media dir       : {MEDIA.resolve()}')
print(f'Log root        : {Path(LOG_ROOT).resolve()}')

### Training Setup
Build TrainConfig from the registered task and apply notebook overrides. Follows the same pattern as `01_walk.ipynb` (uses `TrainConfig.from_task` + `run_train`, not subprocess).

In [ ]:
import datetime, os
from pathlib import Path
from mjlab.scripts.train import TrainConfig

cfg = TrainConfig.from_task(TASK_ID)
cfg.env.scene.num_envs    = NUM_ENVS
cfg.agent.max_iterations  = MAX_ITERATIONS
cfg.agent.logger          = 'tensorboard'
cfg.agent.experiment_name = EXPERIMENT_HINT
cfg.agent.resume          = False

log_dir = (
    Path(LOG_ROOT) / EXPERIMENT_HINT
    / datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
)

# Avoid mutating CUDA_VISIBLE_DEVICES after torch import (triggers "CUDA unknown error"); pin the device in the shell before launch instead.
os.environ.setdefault('MUJOCO_GL', 'egl')

mode = 'RESUME' if RESUME else ('WARM_START_FROM_REACH' if WARM_START_FROM_REACH else 'FROM SCRATCH')
print(f'Task            : {TASK_ID}')
print(f'Envs / iters    : {cfg.env.scene.num_envs} / {cfg.agent.max_iterations}')
print(f'Logger          : {cfg.agent.logger}')
print(f'CUDA device     : {os.environ.get("CUDA_VISIBLE_DEVICES", "(inherited)")}')
print(f'Mode            : {mode}')
print(f'Log dir         : {log_dir.resolve()}')


### Carry Terminal-State Buffer
The drop reset event samples from a buffer of carry-terminal (robot + held box) states so each episode starts already holding the box. Drop a pre-generated `carry_terminal_states.npz` into `outputs/` (fields: `root_z`, `quat_zy`, `joint_pos`, `joint_vel`, `box_pos_by`, `box_quat_by`, `box_lin_vel_by`, `box_ang_vel_by`) before running training. The buffer file is not tracked by git.

In [ ]:
import sys, os
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'LICENSE').exists())
os.chdir(root)
sys.path.insert(0, str(root))
from src.policy_drop import CARRY_BUFFER_PATH

BUFFER_PATH = Path(CARRY_BUFFER_PATH).resolve()
REQUIRED_FIELDS = (
    'root_z', 'quat_zy', 'joint_pos', 'joint_vel',
    'box_pos_by', 'box_quat_by', 'box_lin_vel_by', 'box_ang_vel_by',
)

if not BUFFER_PATH.exists():
    raise FileNotFoundError(
        f'Carry terminal-state buffer missing at {BUFFER_PATH}.\n'
        f'Drop a carry_terminal_states.npz there with fields {REQUIRED_FIELDS} '
        f'(e.g. generated by src.collect_buffer rolling out a trained carry policy) '
        f'before running training.'
    )

import numpy as np
_arr = np.load(BUFFER_PATH)
_missing = [k for k in REQUIRED_FIELDS if k not in _arr.files]
if _missing:
    raise ValueError(
        f'Buffer at {BUFFER_PATH} is missing required fields {_missing}; '
        f'got {sorted(_arr.files)}.'
    )
print(f'[reuse] carry buffer at {BUFFER_PATH}')
print(f'        {_arr["root_z"].shape[0]} snapshots, fields = {sorted(_arr.files)}')

### Run Training
Blocks the kernel until PPO training finishes. Use "Interrupt Kernel" to stop.

In [ ]:
import sys, os
from pathlib import Path
for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (p / 'LICENSE').exists():
        os.chdir(p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from src.train_loop import run as run_train_loop
from mjlab.utils.os import get_checkpoint_path

existing_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
if REUSE_TRAINING and existing_ckpt is not None:
    print(f'[REUSE] Checkpoint found, skipping training.\n        {existing_ckpt}')
else:
    # RESUME loads the last drop checkpoint (matching shapes); WARM_START_FROM_REACH shape-filters the carry checkpoint (only actor/critic first layer re-initialised).
    resume_ckpt, warm_ckpt = None, None
    if RESUME:
        resume_ckpt = get_checkpoint_path(
            log_dir.parent, r'\d{4}-\d{2}-\d{2}_.*', 'model_.*.pt',
        )
    elif WARM_START_FROM_REACH:
        warm_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint=REACH_EXPERIMENT_HINT)
        if warm_ckpt is None:
            raise RuntimeError(
                f'WARM_START_FROM_REACH=True but no reach checkpoint found in '
                f'{LOG_ROOT}/{REACH_EXPERIMENT_HINT}. Run 04_reach.ipynb first.'
            )

    run_train_loop(
        task_id=TASK_ID,
        cfg=cfg,
        log_dir=log_dir,
        warm_start_ckpt=warm_ckpt,
        resume_ckpt=resume_ckpt,
        reset_std=RESET_STD_ON_RESUME if resume_ckpt is not None else None,
    )


### Training Curves & Robustness
Post-training analysis in two blocks: (1) training curves parsed from the TensorBoard event file (reward, PPO diagnostics, per-term reward decomposition, terminations, command tracking); (2) robustness rollouts of the trained policy under lateral pushes and a commanded-velocity sweep, following the RSL "Learning to Walk in Minutes" protocol (Rudin et al., CoRL 2022). Set `RUN_ROBUSTNESS = False` in the setup cell below to skip figures 5 to 6 (the rollout-based ones).

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'LICENSE').exists())
os.chdir(root)
sys.path.insert(0, str(root))
from src.utils import find_latest_tb_log, find_latest_checkpoint

# False skips Figs 5-6 (rollout-based).
RUN_ROBUSTNESS = True

COLORS = {
    'primary':   '#0072B2',
    'accent':    '#D55E00',
    'ok':        '#009E73',
    'warn':      '#E69F00',
    'neutral':   '#56B4E9',
    'text':      '#1A1A1A',
    'grid':      '#B0B0B0',
}

matplotlib.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['Times', 'Nimbus Roman', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 9, 'axes.labelsize': 9, 'legend.fontsize': 7.5,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.linewidth': 0.6, 'axes.edgecolor': COLORS['text'],
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'grid.color': COLORS['grid'], 'grid.linewidth': 0.4, 'grid.alpha': 0.35,
    'figure.dpi': 120,
})

def load_scalar(log_dir, tag):
    """Load a scalar time-series from a TensorBoard event file.
    Args: log_dir: path to the TB log directory; tag: scalar tag name.
    Returns: tuple (steps, values) as numpy arrays (empty if tag missing).
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    if tag not in ea.Tags().get('scalars', []):
        return np.array([]), np.array([])
    evs = ea.Scalars(tag)
    return (np.array([e.step for e in evs], dtype=float),
            np.array([e.value for e in evs], dtype=float))

def list_scalars(log_dir, prefix=''):
    """List scalar tags available in a TensorBoard log, optionally filtered by prefix.
    Args: log_dir: path to the TB log directory; prefix: string filter.
    Returns: list of matching scalar tag names.
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    return [t for t in ea.Tags().get('scalars', []) if t.startswith(prefix)]

def smooth(x, w=15):
    """Apply a centered moving-average filter of window `w` to a 1-D signal.
    Args: x: input array; w: window length in samples.
    Returns: smoothed numpy array with the same length as `x`.
    """
    if len(x) < w or w < 2:
        return np.asarray(x, dtype=float)
    pad = w // 2
    xp = np.pad(x, pad, mode='edge')
    return np.convolve(xp, np.ones(w) / w, mode='valid')[:len(x)]

def apply_grid(ax):
    """Enable major/minor grid with auto minor locators on a matplotlib axis.
    Args: ax: matplotlib Axes to style.
    Returns: none.
    """
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(True, which='major', linewidth=0.5, alpha=0.80)
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.70)
    for s in ax.spines.values():
        s.set_linewidth(0.6)

TB_DIR = find_latest_tb_log(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
assert TB_DIR is not None, 'No TensorBoard log found, train first.'
print(f'TB dir : {TB_DIR}')


### Return & PPO Diagnostics
Plots training curves from the TensorBoard event file: mean reward, episode length, reward gain dR/di, entropy, value loss, and policy std. Depends on helpers defined above.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.2))

x_r, y_r = load_scalar(TB_DIR, 'Train/mean_reward')
if len(y_r):
    axes[0, 0].plot(x_r, smooth(y_r), color=COLORS['primary'], lw=1.2)
axes[0, 0].set_xlabel('PPO iteration')
axes[0, 0].set_ylabel('Mean episode reward')
apply_grid(axes[0, 0])

x, y = load_scalar(TB_DIR, 'Train/mean_episode_length')
if len(y):
    axes[0, 1].plot(x, smooth(y), color=COLORS['ok'], lw=1.2)
axes[0, 1].axhline(130, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 1].text(x[-1] if len(x) else 0, 132, 'max horizon',
                ha='right', va='bottom', fontsize=7, color=COLORS['text'], alpha=0.7)
axes[0, 1].set_xlabel('PPO iteration')
axes[0, 1].set_ylabel('Mean episode length (steps)')
apply_grid(axes[0, 1])

if len(y_r) > 1:
    r_smooth = smooth(y_r)
    dr = np.gradient(r_smooth, x_r)
    axes[0, 2].plot(x_r, smooth(dr, w=25), color=COLORS['warn'], lw=1.2)
    axes[0, 2].axhline(0, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 2].set_xlabel('PPO iteration')
axes[0, 2].set_ylabel(r'Reward gain $dR/di$')
apply_grid(axes[0, 2])

for ax, tag, ylabel, color, log in [
    (axes[1, 0], 'Loss/entropy',    r'Entropy $H(\pi)$',        COLORS['primary'], False),
    (axes[1, 1], 'Loss/value',      r'Value loss $L_V$',        COLORS['accent'],  True),
    (axes[1, 2], 'Policy/mean_std', r'Policy std $\sigma_\pi$', COLORS['ok'],      False),
]:
    x, y = load_scalar(TB_DIR, tag)
    if len(y):
        ax.plot(x, smooth(y), color=color, lw=1.2)
    ax.set_xlabel('PPO iteration')
    ax.set_ylabel(ylabel)
    if log:
        ax.set_yscale('log')
    apply_grid(ax)

fig.tight_layout()
plt.savefig(MEDIA / 'pick_fig1_training.png', dpi=150, bbox_inches='tight')
plt.show()

if len(y_r):
    print(f'Peak reward: {np.max(smooth(y_r)):.2f}')


### Run Viewer
Blocks the kernel while the viewer is open. Use "Interrupt Kernel" to stop.

In [ ]:
import sys, os
from pathlib import Path
for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (p / 'LICENSE').exists():
        os.chdir(p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from mjlab.scripts.play import PlayConfig, run_play

PLAY_AGENT          = 'trained'
PLAY_NUM_ENVS       = 1
PLAY_VIEWER         = 'viser'
CHECKPOINT_OVERRIDE = ''

checkpoint = (
    Path(CHECKPOINT_OVERRIDE)
    if CHECKPOINT_OVERRIDE.strip()
    else find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
)

if PLAY_AGENT == 'trained' and checkpoint is None:
    raise RuntimeError('No checkpoint found. Train first (Cell 5) or set CHECKPOINT_OVERRIDE.')

play_cfg = PlayConfig(
    agent=PLAY_AGENT,
    checkpoint_file=str(checkpoint) if (PLAY_AGENT == 'trained' and checkpoint) else None,
    num_envs=PLAY_NUM_ENVS,
    viewer=PLAY_VIEWER,
)

print(f'Agent      : {PLAY_AGENT}')
print(f'Viewer     : {PLAY_VIEWER}')
print(f'Num envs   : {PLAY_NUM_ENVS}')
if checkpoint:
    print(f'Checkpoint : {checkpoint}')


if RUN_VIEWER:
    print('Viewer starting, open http://localhost:8080 in your browser (viser).')
    print('Press Interrupt Kernel in the toolbar to close it.')
    run_play(TASK_ID, play_cfg)
else:
    print('Set RUN_VIEWER=True to open the trained policy.')